# Churn — exploration & training

Mon premier modèle, bricolé en local. Ça marche, j'ai pas eu le temps de nettoyer.

In [2]:
# imports + tout en vrac
import pandas as pd
import numpy as np
import os, pickle
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, precision_score, recall_score, f1_score

df = pd.read_csv('../data/telco_churn.csv')
print(df.shape)
df.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
# preprocessing... TotalCharges il y a des espaces vides il parait
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
df = df.drop(columns=['customerID'])

y = (df['Churn'] == 'Yes').astype(int)
X = df.drop(columns=['Churn'])

# je split direct, 80/20
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges', 'SeniorCitizen']
cat_cols = [c for c in X.columns if c not in num_cols]

preprocess = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols),
])

model = Pipeline([
    ('prep', preprocess),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)),
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print('accuracy', accuracy_score(y_test, y_pred))
print('precision', precision_score(y_test, y_pred))
print('recall', recall_score(y_test, y_pred))
print('f1', f1_score(y_test, y_pred))
print('roc_auc', roc_auc_score(y_test, y_proba))
print('---')
print(classification_report(y_test, y_pred))

accuracy 0.7938877043354655
precision 0.6390728476821192
recall 0.516042780748663
f1 0.5710059171597633
roc_auc 0.8299770669510433
---
              precision    recall  f1-score   support

           0       0.84      0.89      0.86      1033
           1       0.64      0.52      0.57       374

    accuracy                           0.79      1407
   macro avg       0.74      0.71      0.72      1407
weighted avg       0.78      0.79      0.79      1407



In [4]:
# bon je sauvegarde, j'enverrai le pkl par mail à Marc
os.makedirs('../models', exist_ok=True)
with open('../models/best_model.pkl', 'wb') as f:
    pickle.dump(model, f)
print('ok ->', os.path.abspath('../models/best_model.pkl'))

ok -> c:\Users\Administrateur\Desktop\churn-guard\models\best_model.pkl


todo: refaire ça propre quand j'aurai le temps